# The smell pathway: ORNs to Kenyon cells to MBONs

The olfactory pathway of the fly runs from the olfactory receptor neurons (`ORN_*`, 53 glomerular
types, one per antennal-lobe glomerulus) through the uniglomerular projection neurons
(`<glomerulus>_lPN`/`_adPN`/`_vPN`) to the Kenyon cells (`KC*`, the mushroom body's ~4,000-neuron
expansion layer) and on to the mushroom-body output neurons (`MBON*`, 36 types). In a real fly,
Kenyon-cell activity is famously **sparse**: in vivo, a single odor activates roughly **5-10% of
the Kenyon population** (Honegger, Campbell & Turner 2011, *Cell*; Turner, Bazhenov & Loesel 2008,
*J. Neurophysiol*).

This notebook builds odors as sets of ORN types, drives them at several strengths, and measures
what happens at each stage of that pathway in the model.

The answer is a **negative result**, and a clear one. It comes in three parts:

1. **The first relay works.** Injecting an ORN type drives its glomerular projection neurons up
   from their measured blank rest floors (0.249-0.332 fired per step, 12.4-16.6 Hz; per-odor values in
   the resting-state cell) toward the model's 50 Hz ceiling.
2. **The Kenyon-cell layer is saturated at rest.** In the model's calibrated configuration, the
   blank runs show **4,061 of 4,064 Kenyon cells spiking in every one of the 55 measured steps**
   on the median seed (six of the eight seeds show 4,061-4,062; seed 7 manages 3,402 and seed 4
   only 1,809 -- per-seed, per-cell counts printed in the resting-state cell). The
   repository's own `notebooks/reservoir.ipynb` records the same thing ("Kenyon cells active at a
   given step: 100.0%" for the connectome), and gets chance-level (9.2%) digit readouts from
   Kenyon cells.
3. **So the smell signal vanishes at the Kenyon stage.** In seven of eight noise seeds, odor
   stimulation changes Kenyon-cell spiking by -12 to +24 net spikes across the whole
   population in the 55-step measurement window -- at most 0.0107% of the 223,520 Kenyon spike-slots (24 spikes) --
   at every strength from 0.02 V (where the PN rate is +0.0057 to +0.0172 above its floor) to 2.0 V (the injected ORNs already fire 0.91 per step at 1.0 V, and 2.0 V matches that); the overlap between any two odors' Kenyon "patterns" is a
   Jaccard of 1.0000 ± 0.0000 (min 1.0000, max 1.0000) at every strength. (Seed 4 sits on a tipping point -- see below.) The MBON section says what the
   output neurons do, and the rewired-brain section compares the odor with a second odor and with
   a same-size non-olfactory injection, because a raw change on the rewired brain is not by itself
   an odor at the Kenyon cells.

The model therefore does **not** reproduce the 5-10% sparse Kenyon code. We do not retune any
parameter to make it appear (the issue forbids tuning to hit the number); we show instead how the
result behaves across input strength -- two strengths at the bottom of the range (0.02 V, where
the PN rate is +0.0057 to +0.0172 above its floor; 0.05 V, +0.0443 to +0.0580 above it), fine steps across the rise,
and 1.0 V, where the injected ORNs are already at 0.91 fired
per step so 2.0 V adds nothing (its per-seed Kenyon lists are identical to 1.0 V's); typical-seed
Kenyon effects stay at or below 24 net spikes (0.0107% of the 223,520 spike-slots) throughout,
and the only strong response is that tipping seed, not a code -- and
the saturation is a property of the real wiring: on a degree-preserving rewiring of the same model, resting Kenyon
activity drops to 5.4% ± 0.1 per step in this notebook's window -- measured over the same 55 steps the
sweep uses, on all eight seeds. The reservoir notebook's 18-27% for random wirings is not a
contradiction: it is a different quantity (a 100 ms decayed trace), and the cell that reproduces
it reads 24.4% ± 0.3 on these same rewired weights (see the wiring section).

**Before running:** `pip install flybrain matplotlib scipy`. The first `FlyBrain()` downloads the
brain (~260 MB) once. Everything runs on the CPU.
Plots and the number tables are saved next to this notebook, in `notebooks/smell/`
(`pn_response.png`, `kc_extras.png`, `saturation_control.png`, `pn_response.csv`,
`kc_extras.csv`, `kc_fraction.csv`, `jaccard.csv`, `mbon_sign_test.csv`, `mbon_ttest.csv`,
`rewired_null.csv`), whether it is run from `notebooks/` or from the repository root.
Saved-path lines are relative to the repository (`notebooks/smell/...`).

**No claims about learning:** flybrain has no plasticity in the mushroom body; nothing here says
anything about odor learning, only about the feedforward sweep ORN -> PN -> KC -> MBON.


In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import platform

import flybrain
from flybrain import FlyBrain

# outputs go next to this notebook, whether it is run from notebooks/ or from the repo root
HERE = next(p for p in (Path.cwd(), Path.cwd() / "notebooks") if (p / "smell.ipynb").exists())
OUT = HERE / "smell"
OUT.mkdir(exist_ok=True)


def saved(path):
    '''Print the written file relative to the repository root.'''
    print("saved", f"notebooks/smell/{Path(path).name}")


# sensory_input=False removes every synapse ONTO sensory neurons. The repository's
# olfactory work (notebooks/reservoir.ipynb) uses the same setting. The last code
# cell measures the resting ORN rate with the default True. The ceiling is one spike
# per step; that cell reports the rate as a fraction of the ceiling.
brain = FlyBrain(device="cpu", sensory_input=False)
ct = brain.cell_type.astype(str)

orn_types = sorted(np.unique(ct[np.char.startswith(ct, "ORN_")]))
kc = np.flatnonzero(np.char.startswith(ct, "KC"))
mbons = np.flatnonzero(np.char.startswith(ct, "MBON"))
print(f"flybrain {flybrain.__version__} on {platform.system()} ({platform.machine()}): "
      f"{brain.n:,} neurons; {len(orn_types)} ORN types, {len(kc)} Kenyon cells, "
      f"{len(np.unique(ct[mbons]))} MBON types ({len(mbons)} neurons)")


flybrain 0.1.1 on Darwin (arm64): 166,700 neurons; 53 ORN types, 4064 Kenyon cells, 36 MBON types (97 neurons)


## Odors

Three single glomeruli (`ORN_DM1`, `ORN_DL5`, `ORN_VA2`, each tens of receptor neurons on both
sides) and two seeded random mixtures (4 and 10 glomeruli). A 2 s trial (100 steps of 20 ms) with
the odor on for the middle 1 s; the injected voltage is added to every ORN of the chosen types at
every step while on. Nine strengths: **0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0 and 2.0 V per
step** -- two strengths at the bottom of the range (0.02 V: PN rate +0.0057 to +0.0172 above its floor;
0.05 V: +0.0443 to +0.0580 above it), fine steps across the rise, and 2.0 V kept only to show the ceiling: at 1.0 V the injected ORNs
already fire 0.91 per step, and 2.0 V's per-seed Kenyon extras are identical to 1.0 V's. Eight
noise seeds per condition, and one matched-noise blank run per seed (same noise, no odor) so every
"extra" below is odor minus blank, not odor minus nothing. The blanks double as the resting-state
measurement: per-seed Kenyon rest activity, the count of Kenyon cells that spike in every
measured step, and each odor's projection-neuron rest floor all come from those same 8 runs.

For each odor we also name its **projection neurons** (`<glomerulus>_lPN` / `_adPN` / `_vPN`) --
the uniglomerular cells that in a real fly copy one glomerulus' ORN input to the mushroom body and
lateral horn.


In [2]:
rng = np.random.default_rng(7)
others = [t for t in orn_types if t not in ("ORN_DM1", "ORN_DL5", "ORN_VA2")]
mix4 = [others[i] for i in rng.permutation(len(others))[:4]]
mix10 = [others[i] for i in rng.permutation(len(others))[:10]]
ODORS = {"DM1": ["ORN_DM1"], "DL5": ["ORN_DL5"], "VA2": ["ORN_VA2"], "mix4": mix4, "mix10": mix10}
STRENGTHS = [0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 2.0]
SEEDS = list(range(8))
T, ON, TAIL = 100, (25, 75), 5          # odor on for 1 s; +5 steps of tail for propagation
WINDOW = ON[1] - ON[0] + TAIL           # 55 measured steps: the sweep's window, start to end


def pns_for(types):
    '''Uniglomerular projection neurons of a mixture's glomeruli.'''
    names = [t.removeprefix("ORN_") for t in types]
    return brain.cells([f"{g}_{s}" for g in names for s in ("lPN", "adPN", "vPN")])


def slot_of(idx):
    s = np.full(brain.n, -1, np.int64)
    s[idx] = np.arange(len(idx))
    return s


kc_slot, mbon_slot = slot_of(kc), slot_of(mbons)
mbon_ct = ct[mbons]
pn_of = {name: pns_for(types) for name, types in ODORS.items()}

for name, types in ODORS.items():
    idx = brain.cells(types)
    print(f"{name:6s}: {len(types):2d} ORN types ({len(idx):3d} neurons) -> {len(pn_of[name]):2d} uniglomerular PNs")


def trial(seed, idx, amount, pn_idx=None):
    '''One 2 s run. Per-step firing fractions over the window + tail for the injected
    ORNs, their PNs and the Kenyon cells; one full-brain spike-count array (slice it per
    population: Kenyon cells, MBONs, or any odor's projection neurons).'''
    brain.reset(seed)
    pn_idx = np.asarray(pn_idx if pn_idx is not None else [])
    orn_slot = slot_of(idx) if len(idx) else None
    pn_slot = slot_of(pn_idx) if len(pn_idx) else None
    out = {k: np.zeros(WINDOW) for k in ("orn", "pn", "kc")}
    counts = np.zeros(brain.n, np.int32)
    i = 0
    for t in range(T):
        fired = brain.step(inject=[(idx, amount)] if ON[0] <= t < ON[1] else ())
        if ON[0] <= t < ON[1] + TAIL:
            if len(fired):
                if orn_slot is not None:
                    h = orn_slot[fired]; out["orn"][i] = (h >= 0).sum() / len(idx)
                if pn_slot is not None:
                    h = pn_slot[fired]; out["pn"][i] = (h >= 0).sum() / max(1, len(pn_idx))
                h = kc_slot[fired]; out["kc"][i] = (h >= 0).sum() / len(kc)
                counts[fired] += 1
            i += 1
    return out, counts

DM1   :  1 ORN types ( 74 neurons) ->  2 uniglomerular PNs
DL5   :  1 ORN types ( 43 neurons) ->  2 uniglomerular PNs
VA2   :  1 ORN types ( 83 neurons) ->  2 uniglomerular PNs
mix4  :  4 ORN types (252 neurons) -> 16 uniglomerular PNs
mix10 : 10 ORN types (436 neurons) -> 43 uniglomerular PNs


## First, the resting state

Before any odor: how active is each stage? The 8 blank runs (one per seed, matched noise, no odor)
answer every rest question in this notebook at once: the Kenyon rest fraction per seed, the number
of Kenyon cells that spike in **every one of the 55 measured steps** (a per-cell count, not a
mean over steps), and each odor's projection-neuron rest floor, read from the same full-brain
spike counts. These same runs are the blanks every "extra" below is measured against, so nothing
is measured twice.

In [3]:
t0 = time.time()
blank = {seed: trial(seed, np.empty(0, np.int64), 0.0) for seed in SEEDS}
print(f"{len(blank)} blank (rest) runs in {(time.time() - t0) / 60:.2f} min")
kc_frac = np.array([blank[s][0]["kc"].mean() for s in SEEDS])
print("Kenyon cells at rest, per seed:")
for s in SEEDS:
    every = int((blank[s][1][kc] == WINDOW).sum())
    print(f"  seed {s}: {100 * kc_frac[s]:6.2f}% fire per step; "
          f"{every} of {len(kc)} ({100 * every / len(kc):.2f}%) spike in all {WINDOW} steps")
print(f"median seed: {100 * np.median(kc_frac):.2f}% per step")
print("projection-neuron rest floors (fired/step, mean over seeds, from the same 8 runs):")
pn_floor = {}
for name in ODORS:
    pn_floor[name] = float(np.mean([blank[s][1][pn_of[name]].mean() / WINDOW for s in SEEDS]))
    print(f"  {name:6s}: {pn_floor[name]:.3f} per step  (~{50 * pn_floor[name]:.1f} Hz, "
          f"{len(pn_of[name])} PNs)")
spikes = [len(brain.step()) for _ in range(20)]
print(f"whole-brain spikes per step at rest: ~{np.mean(spikes):.0f}")

8 blank (rest) runs in 0.05 min
Kenyon cells at rest, per seed:
  seed 0:  99.97% fire per step; 4061 of 4064 (99.93%) spike in all 55 steps
  seed 1:  99.97% fire per step; 4062 of 4064 (99.95%) spike in all 55 steps
  seed 2:  99.97% fire per step; 4061 of 4064 (99.93%) spike in all 55 steps
  seed 3:  99.97% fire per step; 4061 of 4064 (99.93%) spike in all 55 steps
  seed 4:  93.02% fire per step; 1809 of 4064 (44.51%) spike in all 55 steps
  seed 5:  99.97% fire per step; 4061 of 4064 (99.93%) spike in all 55 steps
  seed 6:  99.97% fire per step; 4061 of 4064 (99.93%) spike in all 55 steps
  seed 7:  99.60% fire per step; 3402 of 4064 (83.71%) spike in all 55 steps
median seed: 99.97% per step
projection-neuron rest floors (fired/step, mean over seeds, from the same 8 runs):
  DM1   : 0.326 per step  (~16.3 Hz, 2 PNs)
  DL5   : 0.332 per step  (~16.6 Hz, 2 PNs)
  VA2   : 0.249 per step  (~12.4 Hz, 2 PNs)
  mix4  : 0.275 per step  (~13.8 Hz, 16 PNs)
  mix10 : 0.300 per step  (~15.

## Run the odor sweep

Five odors x nine strengths x eight seeds, each measured against its matched-noise blank from the
cell above.

In [4]:
t0 = time.time()
res = {}
for name, types in ODORS.items():
    idx = brain.cells(types)
    for a in STRENGTHS:
        for seed in SEEDS:
            res[name, a, seed] = trial(seed, idx, a, pn_of[name])
print(f"{len(res)} odor runs (+{len(blank)} blanks) in {(time.time() - t0) / 60:.1f} min")

360 odor runs (+8 blanks) in 1.5 min


## The first relay works: projection neurons

Injected ORNs follow the drive, and their uniglomerular projection neurons follow them -- once the
drive is strong enough to leave the rest floor. The blanks put that floor at 0.249-0.332 fired/step
(12.4-16.6 Hz; the table's blank rows are each odor's own projection neurons, measured in the
resting-state cell): at 0.02 V the PN rate is +0.0057 to +0.0172 above its floor, and at 0.05 V it
is +0.0443 to +0.0580 above it -- which is why strengths below 0.1 V are in the sweep; from
0.1 V the odor's glomeruli rise more clearly (0.37-0.44 per step, +0.0994 to +0.1216 above their floors); the
mid range is not monotonic -- DM1's PN rate steps through 0.43, 0.50, 0.56, 0.48, 0.53 at
0.1-0.7 V, and the other odors behave similarly (regime changes in how often the ORNs fire, not a
smooth curve) -- so the sweep samples it finely; and from 1.0 V on the rate pins near the
one-spike-per-step ceiling (0.92-0.93 per step), where the injected ORNs themselves fire 0.91
per step. Whatever the exact shape, the signal crosses ORN -> PN strongly, on the odor's own
glomeruli.


In [5]:
import csv

pn_rows = []
print(f"{'odor':6s} {'V/step':>6s} {'ORN fired/step':>15s} {'PN fired/step':>17s}")
for name in ODORS:
    for a in STRENGTHS:
        orn = np.array([res[name, a, s][0]["orn"].mean() for s in SEEDS])
        pn = np.array([res[name, a, s][0]["pn"].mean() for s in SEEDS])
        print(f"{name:6s} {a:6.2f} {orn.mean():9.2f}±{orn.std(ddof=1):5.2f} "
              f"{pn.mean():11.2f}±{pn.std(ddof=1):5.2f}")
        pn_rows.append((name, a, orn.mean(), orn.std(ddof=1), pn.mean(), pn.std(ddof=1)))
print("blank (rest), each odor's own PNs, from the same 8 blank runs:")
for name in ODORS:
    print(f"  {name:6s} {pn_floor[name]:.3f} per step  (~{50 * pn_floor[name]:.1f} Hz)")
print("PN mean minus that odor's blank floor:")
for a in (0.02, 0.05, 0.1):
    offs = []
    for name in ODORS:
        pn = np.mean([res[name, a, s][0]["pn"].mean() for s in SEEDS])
        offs.append(pn - pn_floor[name])
    print(f"  {a:.2f} V: {min(offs):+.4f} to {max(offs):+.4f}")
with open(OUT / "pn_response.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["odor", "v_per_step", "orn_per_step_mean", "orn_per_step_sd",
                "pn_per_step_mean", "pn_per_step_sd"])
    w.writerows([(n, a, round(o, 4), round(os_, 4), round(p, 4), round(ps, 4))
                 for n, a, o, os_, p, ps in pn_rows])
saved(OUT / "pn_response.csv")


odor   V/step  ORN fired/step     PN fired/step
DM1      0.02      0.02± 0.00        0.34± 0.00
DM1      0.05      0.06± 0.00        0.37± 0.01
DM1      0.10      0.12± 0.00        0.43± 0.01
DM1      0.20      0.24± 0.00        0.50± 0.01
DM1      0.30      0.31± 0.00        0.56± 0.03
DM1      0.50      0.45± 0.00        0.48± 0.00
DM1      0.70      0.46± 0.00        0.53± 0.04
DM1      1.00      0.91± 0.00        0.93± 0.00
DM1      2.00      0.91± 0.00        0.93± 0.00
DL5      0.02      0.02± 0.00        0.34± 0.01
DL5      0.05      0.06± 0.00        0.38± 0.01
DL5      0.10      0.12± 0.00        0.44± 0.02
DL5      0.20      0.24± 0.00        0.51± 0.02
DL5      0.30      0.31± 0.00        0.56± 0.04
DL5      0.50      0.45± 0.00        0.48± 0.01
DL5      0.70      0.46± 0.00        0.55± 0.06
DL5      1.00      0.91± 0.00        0.93± 0.01
DL5      2.00      0.91± 0.00        0.93± 0.01
VA2      0.02      0.02± 0.00        0.26± 0.01
VA2      0.05      0.06± 0.00        0.3

In [6]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for name, color in zip(ODORS, ("#e0342c", "#f4a259", "#bc4b51", "#457b9d", "#2a9d8f")):
    pn = [np.mean([res[name, a, s][0]["pn"].mean() for s in SEEDS]) * 50 for a in STRENGTHS]
    sd = [np.std([res[name, a, s][0]["pn"].mean() for s in SEEDS], ddof=1) * 50 for a in STRENGTHS]
    ax.errorbar(STRENGTHS, pn, yerr=sd, marker="o", capsize=3, color=color, label=name)
    ax.axhline(pn_floor[name] * 50, ls="--", c=color, lw=0.8, alpha=0.7)
ax.set_xscale("log")
ax.set_xlabel("injected voltage per step (V, log scale)")
ax.set_ylabel("glomerular PN firing rate (Hz)")
ax.set_title("The odor's own projection neurons respond\n(dashed lines: each odor's blank rest floor)")
ax.legend(fontsize=8)
plt.savefig(OUT / "pn_response.png", dpi=150, bbox_inches="tight")
saved(OUT / "pn_response.png")


saved notebooks/smell/pn_response.png


## The Kenyon layer: saturated, so the odor adds nothing

Now the measurement the issue asks for: **the fraction of Kenyon cells that spike**, per odor,
with the spread across seeds. The next cell prints it. Two columns, both from the odor runs:

* `KC %/step` is the mean of `res[odor, strength, seed][0]["kc"]` over the 55-step window
  (the fraction of Kenyon cells firing on a step), mean ± sd and min-max across the eight seeds.
* `% with ≥1 spike` is the fraction of Kenyon cells whose count in that window is at least 1.

The resting-state cell already showed why these fractions sit at the ceiling on most seeds: on
six of the eight seeds, 4,061-4,062 of 4,064 Kenyon cells spike in all 55 measured blank steps
(seed 7: 3,402; seed 4: 1,809). A fraction that high cannot separate an odor from no odor. The
informative number beside it is the **odor-attributable** spiking: Kenyon spikes in the stimulus
run minus the matched-noise blank run, per seed.

For **seven of the eight seeds the net effect is tiny at every strength**: the cell prints the
per-seed net extras, the median over all eight seeds (that median includes the tipping seed), and
the median over the other seven. The window holds 4,064 x 55 = 223,520 Kenyon spike-slots. The
cell also prints, per seed, the largest number of Kenyon cells that gain even one spike in any
condition, as a count and as a fraction of 4,064. **Seed 4 sits on a tipping point instead**: in that noise trajectory the blank
run's Kenyon layer settles at 93.02% per step rather than 99.97%, and the odor tips it over (the
cell further below identifies the seed and measures exactly what tips). The layer is metastable between
nearly-saturated regimes, not stably sparse. In the in-vivo literature, 5-10% of Kenyon cells
*respond* to an odor; the per-seed gainers printed below are the model's side of that
comparison, on a population that is ~100% active before the odor arrives.


In [7]:
# Fraction of Kenyon cells that spike, per odor, spread across seeds.
# res[...][0]["kc"] is the per-step fraction over the window (issue #49).
tip = int(SEEDS[int(np.argmin(kc_frac))])
print("Kenyon cells firing per step, odor runs (mean of res[...][0]['kc'] over the window):")
print(f"  {'odor':6s} {'V/step':>6s} {'KC %/step':>16s} {'min-max %/step':>18s} {'% with >=1 spike':>18s}")
kc_frac_rows = []
for name in ODORS:
    for a in STRENGTHS:
        per_step = np.array([res[name, a, s][0]["kc"].mean() for s in SEEDS])
        ever = np.array([(res[name, a, s][1][kc] > 0).mean() for s in SEEDS])
        print(f"  {name:6s} {a:6.2f} {100 * per_step.mean():8.2f}±{100 * per_step.std(ddof=1):5.2f} "
              f"{100 * per_step.min():7.2f} to {100 * per_step.max():6.2f} "
              f"{100 * ever.mean():8.2f}±{100 * ever.std(ddof=1):5.2f}")
        kc_frac_rows.append([name, a, per_step.mean(), per_step.std(ddof=1),
                             per_step.min(), per_step.max(), ever.mean(), ever.std(ddof=1)]
                            + per_step.tolist())
print(f"  blank (no odor), per seed: {[f'{100 * v:.2f}' for v in kc_frac]}")
print(f"  blank mean ± sd: {100 * kc_frac.mean():.2f}% ± {100 * kc_frac.std(ddof=1):.2f} per step")
with open(OUT / "kc_fraction.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["odor", "v_per_step", "kc_per_step_mean", "kc_per_step_sd",
                "kc_per_step_min", "kc_per_step_max", "kc_ever_mean", "kc_ever_sd"]
               + [f"seed{s}_per_step" for s in SEEDS])
    w.writerows(kc_frac_rows)
saved(OUT / "kc_fraction.csv")

print(f"{'odor':6s} {'V/step':>6s} {'extra KC spikes, per seed':>34s} {'median':>7s} {'max':>6s}")
kc_rows = []
for name in ODORS:
    for a in STRENGTHS:
        extra = np.stack([res[name, a, s][1][kc] - blank[s][1][kc] for s in SEEDS])
        tot = extra.sum(1)
        print(f"{name:6s} {a:6.2f} {str(tot.tolist()):>34s} {np.median(tot):7.1f} {tot.max():6.0f}")
        kc_rows.append([name, a] + tot.tolist() + [float(np.median(tot)), float(tot.max())])
gainers = [max(int(((res[name, a, s][1][kc] - blank[s][1][kc]) > 0).sum())
               for name in ODORS for a in STRENGTHS) for s in SEEDS]
print(f"largest count of Kenyon cells gaining >= 1 spike in any odor x strength, per seed: {gainers}")
print("  as a fraction of the 4064 Kenyon cells: "
      + ", ".join(f"seed {s} {100 * gainers[s] / len(kc):.2f}%" for s in SEEDS))
typical = [s for s in SEEDS if s != tip]
shown_all, shown_typ, typ_vals = [], [], []
for name in ODORS:
    for a in STRENGTHS:
        tot = np.array([(res[name, a, s][1][kc] - blank[s][1][kc]).sum() for s in SEEDS])
        shown_all.append(float(f"{np.median(tot):.1f}"))
        shown_typ.append(float(f"{np.median(tot[typical]):.1f}"))
        typ_vals.extend(int(tot[s]) for s in typical)
print(f"printed median column over all {len(SEEDS)} seeds (includes tipping seed {tip}): "
      f"{min(shown_all):.1f} to {max(shown_all):.1f}")
print(f"printed median over the other {len(typical)} seeds: "
      f"{min(shown_typ):.1f} to {max(shown_typ):.1f}")
print(f"typical-seed net extras, every odor x strength: {min(typ_vals)} to {max(typ_vals)}")
print(f"largest share gaining >= 1 spike, typical seeds: "
      f"{max(gainers[s] for s in typical)} of {len(kc)} = "
      f"{100 * max(gainers[s] for s in typical) / len(kc):.2f}%")
print(f"largest share gaining >= 1 spike, seed {tip}: "
      f"{gainers[tip]} of {len(kc)} = {100 * gainers[tip] / len(kc):.2f}%")
with open(OUT / "kc_extras.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["odor", "v_per_step"] + [f"seed{s}" for s in SEEDS] + ["median", "max"])
    w.writerows(kc_rows)
saved(OUT / "kc_extras.csv")


Kenyon cells firing per step, odor runs (mean of res[...][0]['kc'] over the window):
  odor   V/step        KC %/step     min-max %/step   % with >=1 spike
  DM1      0.02    99.06± 2.43   93.04 to  99.97   100.00± 0.00
  DM1      0.05    99.06± 2.42   93.08 to  99.97   100.00± 0.00
  DM1      0.10    99.08± 2.38   93.19 to  99.98   100.00± 0.00
  DM1      0.20    99.09± 2.35   93.29 to  99.98   100.00± 0.00
  DM1      0.30    99.10± 2.32   93.37 to  99.98   100.00± 0.00
  DM1      0.50    99.11± 2.30   93.43 to  99.97   100.00± 0.00
  DM1      0.70    99.11± 2.30   93.43 to  99.98   100.00± 0.00
  DM1      1.00    99.15± 2.20   93.72 to  99.98   100.00± 0.00
  DM1      2.00    99.15± 2.20   93.72 to  99.98   100.00± 0.00
  DL5      0.02    99.06± 2.44   93.04 to  99.97   100.00± 0.00
  DL5      0.05    99.06± 2.43   93.07 to  99.97   100.00± 0.00
  DL5      0.10    99.06± 2.42   93.08 to  99.97   100.00± 0.00
  DL5      0.20    99.07± 2.39   93.17 to  99.97   100.00± 0.00
  DL5      0

In [8]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for name, color in zip(ODORS, ("#e0342c", "#f4a259", "#bc4b51", "#457b9d", "#2a9d8f")):
    tot = np.stack([np.stack([res[name, a, s][1][kc] - blank[s][1][kc] for s in SEEDS]).sum(1)
                    for a in STRENGTHS])          # (strength, seed)
    ax.plot(STRENGTHS, np.median(tot, 1), marker="o", color=color, label=f"{name} (median)")
    ax.scatter(np.repeat(STRENGTHS, len(SEEDS)), tot.ravel(), s=6, color=color, alpha=0.45,
               zorder=3)
ax.axhline(0, c="0.6", lw=0.8)
ax.set_xscale("log")
ax.set_yscale("symlog")
ax.set_xlabel("injected voltage per step (V, log scale)")
ax.set_ylabel("extra Kenyon spikes (odor - matched blank)")
ax.set_title("Kenyon extras per seed (dots) and median (lines);\none tipping-point seed, otherwise tiny effects")
ax.legend(fontsize=8)
plt.savefig(OUT / "kc_extras.png", dpi=150, bbox_inches="tight")
saved(OUT / "kc_extras.png")


saved notebooks/smell/kc_extras.png


### Overlap between odors

Two different odors should activate different Kenyon subsets; in vivo their patterns overlap only
weakly. Here the pattern is the set of Kenyon cells with at least one spike in the 55-step window.
The next cell computes the Jaccard overlap of every odor pair at **every strength**, and prints
the mean, the sd, and the min and max across the eight seeds.


In [9]:
names = list(ODORS)
print("Jaccard overlap of Kenyon spike sets (cells with >= 1 spike in the window).")
print(f"Each pair: mean ± sd across {len(SEEDS)} seeds, then min and max.")
jac_rows = []
for a in STRENGTHS:
    print(f"strength {a:.2f} V:")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            ov = []
            for s in SEEDS:
                left = res[names[i], a, s][1][kc] > 0
                right = res[names[j], a, s][1][kc] > 0
                union = (left | right).sum()
                ov.append((left & right).sum() / union if union else np.nan)
            ov = np.asarray(ov, float)
            print(f"  {names[i]:6s} vs {names[j]:6s}: {ov.mean():.4f} ± {ov.std(ddof=1):.4f} "
                  f"(min {ov.min():.4f}, max {ov.max():.4f})")
            jac_rows.append([a, names[i], names[j], ov.mean(), ov.std(ddof=1), ov.min(), ov.max()]
                            + ov.tolist())
with open(OUT / "jaccard.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["v_per_step", "odor_a", "odor_b", "mean", "sd", "min", "max"]
               + [f"seed{s}" for s in SEEDS])
    w.writerows(jac_rows)
saved(OUT / "jaccard.csv")


Jaccard overlap of Kenyon spike sets (cells with >= 1 spike in the window).
Each pair: mean ± sd across 8 seeds, then min and max.
strength 0.02 V:
  DM1    vs DL5   : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  DM1    vs VA2   : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  DM1    vs mix4  : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  DM1    vs mix10 : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  DL5    vs VA2   : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  DL5    vs mix4  : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  DL5    vs mix10 : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  VA2    vs mix4  : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  VA2    vs mix10 : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  mix4   vs mix10 : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
strength 0.05 V:
  DM1    vs DL5   : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  DM1    vs VA2   : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  DM1    vs mix4  : 1.0000 ± 0.0000 (min 1.0000, max 1.0000)
  DM1    vs mix10 : 1.0000 ± 0.0000 (min 1

### The tipping-point seed, isolated

The `max` column above is one seed, not a typical response. Here it is, measured directly.

In [10]:
best = max(((name, a, seed, res[name, a, seed][1][kc] - blank[seed][1][kc])
             for name in ODORS for a in STRENGTHS for seed in SEEDS),
            key=lambda r: r[3].sum())
name, a, seed, extra = best
frac_blank = blank[seed][0]["kc"].mean()
frac_stim = res[name, a, seed][0]["kc"].mean()
print(f"largest odor effect in the whole sweep: odor {name} at {a:.2f} V/step, seed {seed}")
print(f"  that seed's blank run:   {100 * frac_blank:.2f}% of Kenyon cells fire per step")
print(f"  that seed's odor run:    {100 * frac_stim:.2f}% of Kenyon cells fire per step")
print(f"  extra spikes (odor - blank): {int(extra.sum())} across the {WINDOW}-step window")
print(f"  Kenyon cells gaining >= 1 spike: {(extra > 0).sum()} of {len(kc)} "
      f"({100 * (extra > 0).mean():.2f}%)")
print(f"  all other seeds, same condition: "
      f"{[int((res[name, a, s][1][kc] - blank[s][1][kc]).sum()) for s in SEEDS if s != seed]}")


largest odor effect in the whole sweep: odor mix10 at 1.00 V/step, seed 4
  that seed's blank run:   93.02% of Kenyon cells fire per step
  that seed's odor run:    94.49% of Kenyon cells fire per step
  extra spikes (odor - blank): 3277 across the 55-step window
  Kenyon cells gaining >= 1 spike: 1957 of 4064 (48.15%)
  all other seeds, same condition: [17, 16, 13, 18, 21, 17, 17]


## MBONs: which output neurons respond to which odors?

Mushroom-body output neurons receive the Kenyon cells, and at rest they are also nearly
saturated (the cell prints the fraction of MBON spike-slots used before any odor).

A per-neuron **sign test** at a single strength cannot answer the question. With 8 seeds the
best two-sided sign-test p is `binomtest(8, 8, 0.5)`, and Holm correction over the 97 MBON
neurons multiplies it by 97. The cell prints that product: best possible raw p 0.0078, best possible
Holm-adjusted p 0.758. That floor is above 0.05, so a pass
count of zero is fixed before the data are looked at. That test is uninformative, and "0 pass"
is not evidence of no response.

The test that can pass is a two-sided one-sample **t-test** on a stronger summary of the same
runs. For each odor and each MBON, average the odor-minus-blank extra across strengths, one
number per seed. 2.0 V is left out when its MBON counts duplicate 1.0 V, so the ceiling is not
counted twice. Those seed means are the sample. Holm correction is over the 97 MBONs within
that odor. A large, consistent shift can survive that correction, so a zero here is not fixed in advance the way the sign test's zero is. In this sample a small shift stays under that Holm bar: eight seeds, correction over 97 neurons. The cell prints the strongest neuron either way.


In [11]:
from scipy.stats import binomtest, ttest_1samp

blank_mbon = np.stack([blank[s][1][mbons] for s in SEEDS])
print(f"MBON spikes at rest: {blank_mbon.sum(1).mean():.0f} over the window across {len(mbons)} "
      f"neurons = {100 * blank_mbon.sum(1).mean() / (len(mbons) * WINDOW):.0f}% of MBON spike-slots")
best_raw = float(binomtest(len(SEEDS), len(SEEDS), 0.5).pvalue)
best_holm = len(mbons) * best_raw
print(f"per-condition sign test: best possible raw p ({len(SEEDS)} of {len(SEEDS)}) = {best_raw:.4f}; "
      f"best possible Holm-adjusted p over {len(mbons)} neurons = {best_holm:.3f}")
print("that adjusted floor is above 0.05, so this test is uninformative and its pass count is not evidence")
print("(two-sided sign test on odor-minus-blank extras across seeds; Holm within each odor x strength):")


def holm(p):
    order = np.argsort(p, kind="stable")
    adj = np.empty(len(p))
    run = 0.0
    for r, i in enumerate(order):
        run = max(run, (len(p) - r) * p[i])
        adj[i] = min(run, 1.0)
    return adj


n_sig, best, sig_rows = 0, None, []
for name in ODORS:
    for a in STRENGTHS:
        extra = np.stack([res[name, a, s][1][mbons] - blank[s][1][mbons] for s in SEEDS])
        pos, neg = (extra > 0).sum(0), (extra < 0).sum(0)
        n = pos + neg
        p = np.array([binomtest(int(pos[i]), int(n[i]), 0.5).pvalue if n[i] else 1.0
                      for i in range(len(mbons))])
        adj = holm(p)
        n_sig += int((adj < 0.05).sum())
        i = int(np.argmin(p))
        sig_rows.append([name, a, int((adj < 0.05).sum()), int(pos[i]), int(n[i]), float(p[i])])
        if best is None or p[i] < best[0]:
            best = (float(p[i]), name, a, i, int(pos[i]), int(n[i]), extra[:, i].copy())
print(f"  sign tests run: {len(ODORS) * len(STRENGTHS) * len(mbons)}; "
      f"Holm p < 0.05: {n_sig} (uninformative, as above)")
p0, name, a, i, pos, n, ex = best
print(f"  strongest raw sign-test case: {mbon_ct[i]}[{i}] for {name} at {a:.2f} V; "
      f"per-seed extras {ex.tolist()}; +ve in {pos}/{n} seeds, raw p = {p0:.3f}")

dup = all(np.array_equal(res[name, 2.0, s][1][mbons], res[name, 1.0, s][1][mbons])
          for name in ODORS for s in SEEDS)
pool = [a for a in STRENGTHS if not (dup and a == 2.0)]
print(f"2.0 V MBON counts identical to 1.0 V for every odor and seed: {dup}")
print(f"t-test pools strengths {pool}: mean extra per seed, two-sided t-test, "
      f"Holm over the {len(mbons)} MBONs within each odor")
n_sig_t, t_rows, strongest = 0, [], None
for name in ODORS:
    per_seed = np.stack([
        np.mean([res[name, a, s][1][mbons] - blank[s][1][mbons] for a in pool], 0)
        for s in SEEDS
    ])
    with np.errstate(invalid="ignore", divide="ignore"):
        p = np.asarray(ttest_1samp(per_seed, 0.0, alternative="two-sided").pvalue, float)
    p[~np.isfinite(p)] = 1.0
    adj = holm(p)
    passed = np.flatnonzero(adj < 0.05)
    n_sig_t += int(passed.size)
    for i in range(len(mbons)):
        t_rows.append([name, i, str(mbon_ct[i]), float(per_seed[:, i].mean()),
                       float(p[i]), float(adj[i]), int(adj[i] < 0.05)])
    i = int(np.argmin(p))
    rec = (float(adj[i]), name, i, float(per_seed[:, i].mean()), float(p[i]), per_seed[:, i].copy())
    if strongest is None or rec[0] < strongest[0]:
        strongest = rec
    print(f"  {name:6s}: {passed.size} pass Holm; strongest {mbon_ct[i]}[{i}] "
          f"mean extra {per_seed[:, i].mean():+.3f}, raw p {p[i]:.4g}, Holm p {adj[i]:.3f}")
    print(f"           per-seed means {[round(float(v), 3) for v in per_seed[:, i]]}")
print(f"t-tests with Holm p < 0.05: {n_sig_t} of {len(ODORS) * len(mbons)}")
_, nm, i, mean_ex, raw_p, series = strongest
print(f"strongest t-test overall: {mbon_ct[i]}[{i}] for {nm}, mean extra {mean_ex:+.3f}, "
      f"raw p {raw_p:.4g}, Holm p {strongest[0]:.3f}")
print(f"  per-seed means {[round(float(v), 3) for v in series]}")

with open(OUT / "mbon_sign_test.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["odor", "v_per_step", "n_significant_holm", "best_pos", "best_n", "best_raw_p"])
    w.writerows(sig_rows)
saved(OUT / "mbon_sign_test.csv")
with open(OUT / "mbon_ttest.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["odor", "neuron_index", "neuron_type", "mean_extra", "raw_p", "holm_p", "passes"])
    w.writerows(t_rows)
saved(OUT / "mbon_ttest.csv")


MBON spikes at rest: 4853 over the window across 97 neurons = 91% of MBON spike-slots
per-condition sign test: best possible raw p (8 of 8) = 0.0078; best possible Holm-adjusted p over 97 neurons = 0.758
that adjusted floor is above 0.05, so this test is uninformative and its pass count is not evidence
(two-sided sign test on odor-minus-blank extras across seeds; Holm within each odor x strength):


  sign tests run: 4365; Holm p < 0.05: 0 (uninformative, as above)
  strongest raw sign-test case: MBON10[48] for VA2 at 0.30 V; per-seed extras [0, 1, 3, 1, 1, 1, 1, 2]; +ve in 7/7 seeds, raw p = 0.016
2.0 V MBON counts identical to 1.0 V for every odor and seed: True
t-test pools strengths [0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]: mean extra per seed, two-sided t-test, Holm over the 97 MBONs within each odor
  DM1   : 0 pass Holm; strongest MBON10[92] mean extra +0.703, raw p 0.02059, Holm p 1.000
           per-seed means [1.375, 0.5, 0.75, -0.5, 1.25, 1.5, 0.25, 0.5]
  DL5   : 0 pass Holm; strongest MBON04[90] mean extra -0.078, raw p 0.04917, Holm p 1.000
           per-seed means [0.0, -0.25, -0.125, 0.0, -0.125, 0.0, 0.0, -0.125]
  VA2   : 0 pass Holm; strongest MBON10[92] mean extra +0.781, raw p 0.002492, Holm p 0.242
           per-seed means [1.375, 0.625, 0.375, 0.5, 1.125, 1.25, 0.0, 1.0]
  mix4  : 0 pass Holm; strongest MBON33[14] mean extra -0.047, raw p 0.0796, Holm p

## Is the saturation the wiring's fault?

The Kenyon cells being pinned at the ceiling is a property of the **real connectome's** recurrent
drive in this model recipe, not of the recipe alone: the repository's `notebooks/reservoir.ipynb`
records Kenyon activity of 100% for the connectome and much lower for random rewirings of the
same model. We reproduce that control here over all eight seeds (tipping seed included), with a
degree-preserving rewiring: every neuron keeps its in- and out-degree and its synaptic signs; who
connects to whom is random, and incoming weight strengths are rescaled to match. The control
settles for 25 steps and then measures the **same 55-step window the sweep uses** -- the procedure
is identical to the blank runs, so the connectome row below matches the resting-state cell
exactly, seed for seed.

**Why the rewired number here is lower than the reservoir notebook's 18-27%: the two notebooks
report different quantities.** The reservoir's "Kenyon cells active at a given step" is a 100 ms
decayed trace (`tau = 0.1`) read against a 0.5 threshold -- a single spike holds the decaying
trace above 0.5 for four consecutive step-readings (decay `exp(-0.02/0.1)` per step), so a cell
counts as active if it has spiked within roughly the last three steps (the reservoir's own gloss:
"100% spiked within the last 3 steps"). This notebook reports the per-step spike fraction, a
strictly tighter measure. The last cell below measures the reservoir's quantity on these same
rewired weights, same seeds, same window: it reads 24.4% ± 0.3, inside the reservoir's 18-27% band,
while the per-step fraction reads 5.4% ± 0.1. (The reservoir's band pools its two random-wiring
families in a driven digit-stream regime -- its degree-preserving shuffle reads 25.9-27.0% there,
its Erdős–Rényi 18.6-18.7% -- and even at rest, on this single degree-preserving rewiring, its
measure lands in the same pooled band.) There is no contradiction to explain away: both measures
agree the real wiring saturates the Kenyon layer and random wiring does not.

(Swapping the weight matrix uses `FlyBrain`'s internal arrays, exactly as `reservoir.ipynb` does;
it is a notebook trick, not a public API.)


In [12]:
from scipy import sparse

original = sparse.csc_matrix((brain.weights, brain.indices, brain.indptr), shape=(brain.n, brain.n)).tocsr()
row_strength = np.asarray(abs(original).sum(1)).ravel()


def match_input_strength(W):
    W = W.tocsr()
    W.sum_duplicates()
    s = np.asarray(abs(W).sum(1)).ravel()
    scale = np.divide(row_strength, s, out=np.zeros_like(s), where=s > 0)
    W = sparse.diags(scale.astype(np.float32)) @ W
    W.eliminate_zeros()
    return W.tocsr()


coo = original.tocoo()
post = np.random.default_rng(11).permutation(coo.row)
shuffled = match_input_strength(sparse.coo_matrix((coo.data, (post, coo.col)), shape=original.shape))


def use_weights(W):
    W = W.tocsr().astype(np.float32)
    W.sort_indices()
    csc = W.tocsc()
    brain.indptr, brain.indices, brain.weights = csc.indptr, csc.indices, csc.data


def rest_kc_fraction(seeds, settle=25, steps=WINDOW):
    '''The same window the sweep uses: settle for 25 steps, then measure WINDOW steps.'''
    fr = []
    for seed in seeds:
        brain.reset(seed)
        for _ in range(settle):
            brain.step()
        c = 0
        for _ in range(steps):
            fired = brain.step()
            c += (kc_slot[fired] >= 0).sum() if len(fired) else 0
        fr.append(c / (steps * len(kc)))
    return np.array(fr)


frac_real = rest_kc_fraction(SEEDS)
use_weights(shuffled)
frac_shuffled = rest_kc_fraction(SEEDS)
use_weights(original)
print("resting Kenyon cells firing per step, per seed (settle 25, measure 55 -- the sweep's window):")
print("  connectome:", [f"{100 * v:.2f}" for v in frac_real])
print("  blank runs:", [f"{100 * v:.2f}" for v in kc_frac], "(same procedure, resting-state cell)")
print("  rewired:   ", [f"{100 * v:.1f}" for v in frac_shuffled])
print(f"means: connectome {100 * frac_real.mean():.1f}% ± {100 * frac_real.std(ddof=1):.1f}, "
      f"degree-preserving rewiring {100 * frac_shuffled.mean():.1f}% "
      f"± {100 * frac_shuffled.std(ddof=1):.1f}")
print(f"connectome row vs the blank runs (identical procedure): "
      f"max per-seed difference {np.abs(frac_real - kc_frac).max():.1e}")

fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.bar([0, 1], [100 * frac_real.mean(), 100 * frac_shuffled.mean()],
       yerr=[100 * frac_real.std(ddof=1), 100 * frac_shuffled.std(ddof=1)], capsize=4,
       color=["#e0342c", "#457b9d"])
ax.set_xticks([0, 1], ["connectome", "rewired (degree-preserving)"])
ax.set_ylabel("resting Kenyon cells firing per step (%)")
ax.set_title("The saturation is the real wiring's")
plt.savefig(OUT / "saturation_control.png", dpi=150, bbox_inches="tight")
saved(OUT / "saturation_control.png")


resting Kenyon cells firing per step, per seed (settle 25, measure 55 -- the sweep's window):
  connectome: ['99.97', '99.97', '99.97', '99.97', '93.02', '99.97', '99.97', '99.60']
  blank runs: ['99.97', '99.97', '99.97', '99.97', '93.02', '99.97', '99.97', '99.60'] (same procedure, resting-state cell)
  rewired:    ['5.3', '5.4', '5.5', '5.3', '5.4', '5.4', '5.4', '5.5']
means: connectome 99.1% ± 2.4, degree-preserving rewiring 5.4% ± 0.1
connectome row vs the blank runs (identical procedure): max per-seed difference 1.1e-16
saved notebooks/smell/saturation_control.png


In [13]:
from flybrain import Trace


def trace_kc_fraction(W, seeds, settle=25, steps=WINDOW):
    '''The reservoir notebook's measure, on the same weights: a 100 ms decayed trace
    (tau = 0.1); a Kenyon cell counts as active when its trace exceeds 0.5.'''
    use_weights(W)
    tr = Trace(brain, idx=kc, tau=0.1)
    fr = []
    for seed in seeds:
        brain.reset(seed)
        tr.reset()
        for _ in range(settle):
            tr.observe(brain.step())
        c = 0
        for _ in range(steps):
            tr.observe(brain.step())
            c += (tr.features() > 0.5).sum()
        fr.append(c / (steps * len(kc)))
    return np.array(fr)


tr_real = trace_kc_fraction(original, SEEDS)
tr_shuffled = trace_kc_fraction(shuffled, SEEDS)
use_weights(original)
print("the reservoir's own measure (trace tau = 0.1, active = trace > 0.5),")
print("same weights, seeds and window as the control above:")
print(f"  connectome: {100 * tr_real.mean():.1f}% ± {100 * tr_real.std(ddof=1):.1f} "
      f"(reservoir.ipynb records 100.0%)")
print(f"  rewired:   {100 * tr_shuffled.mean():.1f}% ± {100 * tr_shuffled.std(ddof=1):.1f} "
      f"(reservoir.ipynb records 18-27% for random wirings)")
print(f"  the same rewired runs, per-step spike fraction: "
      f"{100 * frac_shuffled.mean():.1f}% ± {100 * frac_shuffled.std(ddof=1):.1f}")

the reservoir's own measure (trace tau = 0.1, active = trace > 0.5),
same weights, seeds and window as the control above:
  connectome: 99.2% ± 2.2 (reservoir.ipynb records 100.0%)
  rewired:   24.4% ± 0.3 (reservoir.ipynb records 18-27% for random wirings)
  the same rewired runs, per-step spike fraction: 5.4% ± 0.1


## The same odor on the rewired brain, against a null

On the connectome the Kenyon layer is saturated, so an odor has almost no silent cells to
recruit. On the degree-preserving rewiring the layer fires 5.4% ± 0.1 per step, and a perturbation
can move spike counts. On an unsaturated network that move is not, by itself, evidence the odor
reached the Kenyon cells: injecting anywhere can knock cells off the matched-noise path.

This cell keeps DM1 at 1.0 V on the rewired weights, all eight seeds, matched-noise blank, and
adds two comparisons on those same weights:

1. **A second odor**, DL5 at 1.0 V. The pattern comparison is the Jaccard of the Kenyon cells
   that spike at least once, and the correlation of the per-cell odor-minus-blank extras.
2. **A same-size non-olfactory injection.** ORN_DM1 is 74 neurons. The null is 74 optic-lobe
   sensory neurons (`superclass == "ol_sensory"`), a fixed draw from `numpy` generator seed 13,
   driven at the same 1.0 V. Optic-lobe sensory neurons are not on the smell pathway.

The changed-cell fraction is read against the null. A fraction that matches the null is
sensitivity to perturbation. A pattern that separates the two odors from the null, and from each
other, is the odor at the Kenyon cells. The connectome's own DM1 numbers are printed beside
these.


In [14]:
dm1_idx = brain.cells(["ORN_DM1"])
dl5_idx = brain.cells(["ORN_DL5"])
visual = np.flatnonzero(brain.superclass.astype(str) == "ol_sensory")
null_idx = np.random.default_rng(13).choice(visual, size=len(dm1_idx), replace=False)
null_idx.sort()
null_types = sorted(set(ct[null_idx].astype(str)))
print(f"null set: {len(null_idx)} optic-lobe sensory neurons "
      f"(ORN_DM1 has {len(dm1_idx)}; pool {len(visual)}; rng seed 13)")
print(f"  {len(null_types)} cell types in the null set, first few: {null_types[:6]}")


def pearson(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / denom) if denom else np.nan


def jaccard_bool(a, b):
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else np.nan


rewired_runs = {}
use_weights(shuffled)
try:
    for seed in SEEDS:
        blank_r = trial(seed, np.empty(0, np.int64), 0.0)
        dm1 = trial(seed, dm1_idx, 1.0, pns_for(["ORN_DM1"]))
        dl5 = trial(seed, dl5_idx, 1.0, pns_for(["ORN_DL5"]))
        null = trial(seed, null_idx, 1.0)
        rewired_runs[seed] = blank_r, dm1, dl5, null
finally:
    use_weights(original)

print(f"{'seed':>4} {'DM1 net':>8} {'DM1%chg':>8} {'DL5 net':>8} {'DL5%chg':>8} "
      f"{'null net':>9} {'null%chg':>8}")
rows = []
bundle = {k: [] for k in ("chg_dm1", "chg_dl5", "chg_null", "net_dm1", "net_dl5", "net_null",
                          "r_odors", "r_null", "jac_spike_odors", "jac_spike_null",
                          "jac_spike_blank", "jac_up_odors", "jac_up_null")}
for seed in SEEDS:
    blank_r, dm1, dl5, null = rewired_runs[seed]
    e1 = dm1[1][kc] - blank_r[1][kc]
    e2 = dl5[1][kc] - blank_r[1][kc]
    e0 = null[1][kc] - blank_r[1][kc]
    n1, n2, n0 = int(e1.sum()), int(e2.sum()), int(e0.sum())
    c1, c2, c0 = (float((e1 != 0).mean()), float((e2 != 0).mean()), float((e0 != 0).mean()))
    r12, r10 = pearson(e1, e2), pearson(e1, e0)
    s1, s2, s0, sb = dm1[1][kc] > 0, dl5[1][kc] > 0, null[1][kc] > 0, blank_r[1][kc] > 0
    j_sp_od = jaccard_bool(s1, s2)
    j_sp_null = jaccard_bool(s1, s0)
    j_sp_bl = jaccard_bool(s1, sb)
    j_up_od = jaccard_bool(e1 > 0, e2 > 0)
    j_up_null = jaccard_bool(e1 > 0, e0 > 0)
    print(f"{seed:4d} {n1:8d} {100 * c1:7.1f}% {n2:8d} {100 * c2:7.1f}% {n0:9d} {100 * c0:7.1f}%")
    for key, val in (("chg_dm1", c1), ("chg_dl5", c2), ("chg_null", c0),
                     ("net_dm1", n1), ("net_dl5", n2), ("net_null", n0),
                     ("r_odors", r12), ("r_null", r10),
                     ("jac_spike_odors", j_sp_od), ("jac_spike_null", j_sp_null),
                     ("jac_spike_blank", j_sp_bl), ("jac_up_odors", j_up_od),
                     ("jac_up_null", j_up_null)):
        bundle[key].append(val)
    rows.append([seed, n1, c1, n2, c2, n0, c0, r12, r10, j_sp_od, j_sp_null, j_sp_bl,
                 j_up_od, j_up_null])

def band(key, scale=1.0, digits=1, suffix=""):
    v = scale * np.asarray(bundle[key], float)
    return (f"{v.mean():.{digits}f}{suffix} ± {v.std(ddof=1):.{digits}f} "
            f"(min {v.min():.{digits}f}, max {v.max():.{digits}f})")

print("changed-cell fraction (spike count differs from the matched blank):")
print(f"  DM1  {band('chg_dm1', scale=100, digits=1, suffix='%')}")
print(f"  DL5  {band('chg_dl5', scale=100, digits=1, suffix='%')}")
print(f"  null {band('chg_null', scale=100, digits=1, suffix='%')}")
print("net extra Kenyon spikes:")
print(f"  DM1  {band('net_dm1', digits=1)}")
print(f"  DL5  {band('net_dl5', digits=1)}")
print(f"  null {band('net_null', digits=1)}")
print("correlation of per-KC odor-minus-blank extras:")
print(f"  DM1 vs DL5  {band('r_odors', digits=3)}")
print(f"  DM1 vs null {band('r_null', digits=3)}")
print("Jaccard of Kenyon cells with >= 1 spike:")
print(f"  DM1 vs DL5   {band('jac_spike_odors', digits=3)}")
print(f"  DM1 vs null  {band('jac_spike_null', digits=3)}")
print(f"  DM1 vs blank {band('jac_spike_blank', digits=3)}")
print("Jaccard of Kenyon cells whose count rose:")
print(f"  DM1 vs DL5  {band('jac_up_odors', digits=3)}")
print(f"  DM1 vs null {band('jac_up_null', digits=3)}")

conn = [(int((res["DM1", 1.0, s][1][kc] - blank[s][1][kc]).sum()),
         100 * (res["DM1", 1.0, s][1][kc] - blank[s][1][kc] != 0).mean()) for s in SEEDS]
print("same odor (DM1 at 1.0 V) on the connectome, same seeds:")
for s, (net, moved) in zip(SEEDS, conn):
    print(f"  seed {s}: net extras {net:+5d}; Kenyon cells with any change {moved:.2f}%")

with open(OUT / "rewired_null.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["seed", "dm1_net", "dm1_changed", "dl5_net", "dl5_changed", "null_net",
                "null_changed", "r_dm1_dl5", "r_dm1_null", "jaccard_spike_dm1_dl5",
                "jaccard_spike_dm1_null", "jaccard_spike_dm1_blank", "jaccard_up_dm1_dl5",
                "jaccard_up_dm1_null"])
    w.writerows(rows)
saved(OUT / "rewired_null.csv")


null set: 74 optic-lobe sensory neurons (ORN_DM1 has 74; pool 6098; rng seed 13)
  4 cell types in the null set, first few: [np.str_('R1-6'), np.str_('R7'), np.str_('R7R8_unclear'), np.str_('R8')]


seed  DM1 net  DM1%chg  DL5 net  DL5%chg  null net null%chg
   0      244    48.5%       40    47.3%       -55    48.3%
   1      -62    47.5%       37    48.5%       145    49.8%
   2      164    48.0%       45    48.8%      -128    49.3%
   3       41    48.2%      125    49.7%        24    46.7%
   4      101    49.1%       40    49.1%       -29    48.4%
   5      102    47.8%      -18    49.8%       -70    48.5%
   6       70    49.1%      155    48.7%       -88    48.6%
   7      199    48.6%       85    48.1%       -55    47.4%
changed-cell fraction (spike count differs from the matched blank):
  DM1  48.4% ± 0.6 (min 47.5, max 49.1)
  DL5  48.8% ± 0.8 (min 47.3, max 49.8)
  null 48.4% ± 1.0 (min 46.7, max 49.8)
net extra Kenyon spikes:
  DM1  107.4 ± 96.2 (min -62.0, max 244.0)
  DL5  63.6 ± 55.3 (min -18.0, max 155.0)
  null -32.0 ± 83.9 (min -128.0, max 145.0)
correlation of per-KC odor-minus-blank extras:
  DM1 vs DL5  0.484 ± 0.018 (min 0.452, max 0.514)
  DM1 vs null 0.495 

## Resting ORN rate with `sensory_input=True`

The odor sweep uses `sensory_input=False`, as `notebooks/reservoir.ipynb` does. With
`sensory_input=True` the ORN-to-ORN synapses are in the circuit. The ceiling in this model is
one spike per step (50 Hz at the 20 ms step). The cell below measures the resting rate of every
ORN and prints it as a fraction of that ceiling: **50% (0.50 spikes per neuron per step)**.
That printed fraction is the number this notebook uses.


In [15]:
brain2 = FlyBrain(device="cpu", sensory_input=True)
ct2 = brain2.cell_type.astype(str)
orn_all = np.flatnonzero(np.char.startswith(ct2, "ORN_"))
orn_slot2 = np.full(brain2.n, -1, np.int64)
orn_slot2[orn_all] = np.arange(len(orn_all))
brain2.reset(0)
for _ in range(10):
    brain2.step()
c = 0
for _ in range(30):
    fired = brain2.step()
    c += (orn_slot2[fired] >= 0).sum() if len(fired) else 0
rate = c / (30 * len(orn_all))
print(f"ORN firing per step with sensory_input=True: {100 * rate:.0f}% "
      f"({rate:.2f} spikes per neuron per step; the ceiling is 1 spike per step)")
print(f"that is {100 * rate:.0f}% of the one-spike-per-step ceiling")
del brain2


ORN firing per step with sensory_input=True: 50% (0.50 spikes per neuron per step; the ceiling is 1 spike per step)
that is 50% of the one-spike-per-step ceiling


## Summary

* **Fraction of Kenyon cells that spike, from the printed table.** Firing per step has mean
  99.06% to 99.25% across odors and strengths (sd 1.93 to 2.44). The seed min-max column runs
  from 93.02% to 99.98%; the low end is seed 4. The fraction with at least one spike in the
  window is 100.00±0.00 for every odor and every strength, and the blank mean is 99.06% ± 2.44
  per step. The Kenyon layer of this model sits at (or a whisker from) the 50 Hz
  ceiling at rest -- on six of the eight seeds, 4,061-4,062 of 4,064 Kenyon cells spike in every
  one of the 55 measured blank steps (seed 7: 3,402; seed 4: 1,809) -- so the sparseness measure cannot even in principle separate odors. One
  seed in eight settles at 93.02% instead of 99.97%: the layer is metastable between near-saturated
  regimes, and the only sizable odor effect in the whole notebook is that seed tipping over.
* **Odor-attributable Kenyon spiking: tiny on the seven typical seeds** -- per-seed net extras of
  -12 to +24 spikes across 4,064 neurons in the 55-step window, at every strength from 0.02 V
  (where the PN rate is +0.0057 to +0.0172 above its floor) to 2.0 V (ORN rate 0.91 per step, the same as at 1.0 V). The
  printed median column over all eight seeds runs from -3.5 to 17.0 and includes the tipping
  seed. The printed median over the other seven seeds runs from -4.0 to 17.0. The largest number of Kenyon cells gaining even one
  spike in any condition is 2-6 cells on the seven typical seeds (at most 0.15% of 4,064) and
  1,957 cells (48.15%) on the tipping seed, so the 0.15% figure is the typical-seed figure only.
  The in-vivo literature reports 5-10% of Kenyon cells responding to an odor (Honegger et al. 2011;
  Turner et al. 2008). On the typical seeds the model is short of that band by a factor of about
  30-70, toward total saturation (at 1.0 V and above, DM1, VA2 and mix10 gain consistently across
  typical seeds -- a real effect, still at most +24 spikes). The exception is the tipping-point seed,
  where the odor knocks a metastable 93.02%-active layer to 94.49% (isolated in its own cell
  above, at its largest: 1,957 of 4,064 Kenyon cells gain a spike, +3,277 net, for the
  10-glomerulus mixture at 1.0 V). That is the only large strength-dependent effect in the sweep,
  and it is a bifurcation rather than a graded code.
* **Pattern overlap between odors.** At every strength from 0.02 V to 2.0 V, every odor pair
  has Jaccard 1.0000 ± 0.0000 (min 1.0000, max 1.0000) across the eight seeds.
* **MBONs.** 91% of their spike-slots are used at rest (4853 spikes over the window across 97 neurons). The per-condition Holm sign test cannot pass at 8 seeds: best possible raw p 0.0078, best possible Holm-adjusted p 0.758, and 0 of 4365 tests pass. That zero is fixed in advance. The t-test pools strengths 0.02 through 1.0 (2.0 V MBON counts duplicate 1.0 V). A large consistent shift can pass it. In this sample 0 of 485 neurons pass, and a small shift stays under that Holm bar. Strongest case: MBON10[92] for VA2, mean extra +0.781, raw p 0.002492, Holm p 0.242.
* **The signal does cross the first relay**: injected ORNs drive their glomerular projection
  neurons off their measured 0.249-0.332 per-step rest floors toward the 50 Hz ceiling (at 0.02 V
  the PN rate is +0.0057 to +0.0172 above its floor; at 0.1 V it is +0.0994 to +0.1216 above it). The failure starts at the PN -> Kenyon stage.
* **Resting saturation is specific to the real wiring.** A degree-preserving rewiring of the
  same model runs its Kenyon layer at 5.4% ± 0.1 per step at rest over the sweep's own 55-step window,
  all eight seeds, against 99.1% ± 2.4 on the connectome. The same rewired weights read 24.4% ± 0.3 under the
  reservoir notebook's decayed-trace measure (connectome trace 99.2% ± 2.2), inside its published 18-27% band for random wirings.
  The odor comparison is a separate measurement on those same rewired weights. DM1 at 1.0 V,
  DL5 at 1.0 V, and a same-size optic-lobe sensory injection (74 neurons, generator seed 13)
  land on the same bands. Changed-cell fraction: DM1 48.4% ± 0.6 (min 47.5, max 49.1), DL5
  48.8% ± 0.8 (min 47.3, max 49.8), null 48.4% ± 1.0 (min 46.7, max 49.8). Correlation of
  per-cell extras: DM1 vs DL5 0.484 ± 0.018 (min 0.452, max 0.514), DM1 vs null 0.495 ± 0.008
  (min 0.480, max 0.506). Jaccard of cells whose count rose: DM1 vs DL5 0.346 ± 0.013
  (min 0.326, max 0.368), DM1 vs null 0.349 ± 0.014 (min 0.326, max 0.368). Jaccard of cells
  with at least one spike: DM1 vs DL5 0.927 ± 0.003 (min 0.922, max 0.932), DM1 vs null
  0.928 ± 0.003 (min 0.924, max 0.933), DM1 vs the matched blank 0.928 ± 0.003 (min 0.923,
  max 0.933).

**No parameter was retuned** to try to recover a sparse code (the issue forbids it); the result
is reported at the repository's own calibrated settings (tonic 0.14, gain 3.0; see
`FlyBrain`'s docstring).

## What the model leaves out

* **No plasticity, no learning claims.** The mushroom body here is a fixed relay; nothing in this
  notebook bears on odor learning, which is the mushroom body's best-known job.
* **No graded synaptic transmission.** Real Kenyon cells receive small, largely subthreshold
  inputs; a 20 ms-step LIF model with normalized weights and gain 3.0 sums whole-population
  volleys instead. The calibration that keeps descending neurons quiet at rest (README finding 3)
  leaves the Kenyon layer saturated.
* **Injection, not an odor plume.** The "odor" is voltage added directly to ORN types at a
  uniform rate; real odors arrive as temporally structured concentration transients across many
  glomeruli.
* **A single fixed weight gain** for every synapse class, no short-term dynamics, 20 ms time
  steps, and 8 noise seeds -- probabilities are over model noise, not over flies.
